In [1]:
import os
from dotenv import load_dotenv

# 強制重新讀取檔案
load_dotenv(override=True) 

# 印出前四碼確認
key = os.getenv("GOOGLE_API_KEY")
if key:
    print(f"成功讀取到金鑰，開頭是: {key[:4]}")
else:
    print("還是讀不到 .env 檔案，請檢查檔案名稱是否真的是 .env (前面有個點)")

還是讀不到 .env 檔案，請檢查檔案名稱是否真的是 .env (前面有個點)


In [2]:
import os
import google.generativeai as genai
from dotenv import load_dotenv

# 1. 放入你的通行證
load_dotenv()
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

# 2. 查詢並印出支援「生成內容 (generateContent)」的模型
print("你的帳號目前可以使用的模型有：")
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

d:\Gemini\Gemini\.venv\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.11) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


你的帳號目前可以使用的模型有：
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro

d:\Gemini\Gemini\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\User\AppData\Local\Temp\ipykernel_7360\3067008785.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [ ]:
import os
import google.generativeai as genai
from dotenv import load_dotenv
from PIL import Image
from docx import Document
from docx.shared import Pt         
from docx.oxml.ns import qn         
# 1. 金鑰設定
load_dotenv()
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

model = genai.GenerativeModel('gemini-2.5-flash')
# 2. 圖片路徑
image_file = "revenue_chart.png" 

print(f"正在讀取圖片 {image_file} ")
try:
    chart_image = Image.open(image_file)
except FileNotFoundError:
    print(f"找不到圖片檔案：{image_file}")
    exit()

# 3. 提示區
prompt = """
你現在是一位專業的財務數據分析師。
請分析附上的這張營收或業績圖表，並為我撰寫一份給管理層看的正式分析報告。

報告必須包含以下完整結構：
一、 標題 (Title)
二、 報告摘要 (Executive Summary)
三、 前言與背景 (Introduction)
四、 數據分析與發現 (Data Analysis)
五、 結論與建議 (Conclusion)

格式要求：
1. 請直接從「一、 標題」開始你的回答，且最前面不要加多餘的文字與圖表名稱
2. 絕對不要使用任何 Markdown 語法（例如 **粗體**、# 標題、- 項目符號等）。
3. 每個段落請盡量保持簡潔，且是純文字的輸出，適合直接放入 Word 報告中。
4. 數字規範：請使用阿拉伯數字（例如 2024、3.5%），不要使用中文數字。
5. 邏輯順序：列舉多個地區數據時，請按照圖表中數據的大小順序從大到小排列，不要跳躍式論述。
"""

print("正在請 Gemini 分析圖表並生成報告")

# 4. 將「提示詞」與「圖片」一起打包發送給模型
response = model.generate_content([prompt, chart_image])
report_text = response.text

print("\nGemini 分析完成\n")
print("內容預覽：")
print(report_text[:150]+"\n") 

# 5. 將生成的內容存成 Word 檔
print("正在將報告儲存為 Word 檔")
doc = Document()

# 簡單處理一下排版，將 AI 生成的文字逐行寫入 Word
start_writing = False # 建立一個開關，確保從第一點才開始寫入

# 逐行清理並寫入 Word
for line in report_text.split('\n'):
    # 把這行文字前後的空白、星號(*)、減號(-)、井字號(#) 都清除
    clean_line = line.strip(" *-#\t") 
    
    if not clean_line:
        continue

    # 檢查是否為大標題開頭
    is_heading = clean_line.startswith('一、') or clean_line.startswith('二、') or clean_line.startswith('三、') or clean_line.startswith('四、') or clean_line.startswith('五、')

    # 如果遇到「一、」，就把寫入開關打開
    if clean_line.startswith('一、'):
        start_writing = True

    # 只有當開關打開時，才把內容寫進 Word
    if start_writing:
        if is_heading:
            # 處理大標題
            heading = doc.add_heading(level=2)
            run = heading.add_run(clean_line)
            
            # 設定標題字體 (需保留您原本的字體匯入與 qn 設定)
            run.font.name = '微軟正黑體'
            run._element.rPr.rFonts.set(qn('w:eastAsia'), '微軟正黑體')
            run.font.size = Pt(16)
            run.font.bold = True
        else:
            # 處理一般內文
            paragraph = doc.add_paragraph()
            
            paragraph.paragraph_format.first_line_indent = Pt(24) 
            
            run = paragraph.add_run(clean_line)
            
            # 設定內文字體與大小 (維持您原本的設定)
            run.font.name = '微軟正黑體'
            run._element.rPr.rFonts.set(qn('w:eastAsia'), '微軟正黑體')
            run.font.size = Pt(12)

# 設定儲存的檔名
folder_path = r"C:\Users\User\OneDrive\桌面\年度分析"
file_name = "年度營收分析報告.docx"
os.makedirs(folder_path, exist_ok=True)
full_output_path = os.path.join(folder_path, file_name)
doc.save(full_output_path)

print(f"儲存為：{full_output_path}")

正在讀取圖片 revenue_chart.png 
正在請 Gemini 分析圖表並生成報告

Gemini 分析完成

內容預覽：
一、 標題
各大區18個品類銷售業績分析報告

二、 報告摘要
本報告分析了公司18個產品類別在北方區、東南區和中西區的銷售業績。數據顯示，部分品類和區域表現突出，尤其是在品類五由中西區主導，以及品類十七由東南區展現強勁實力。北方區則在多個高價值品類中均有穩定貢獻。同時，報告也發現了多個品類銷售業績

正在將報告儲存為 Word 檔
儲存為：C:\Users\User\OneDrive\桌面\年度分析\年度營收分析報告.docx
